
# [The Differentiation Trinity](https://towardsdatascience.com/automatic-differentiation-autodiff-a-brief-intro-with-examples-3f3d257ffe3b/):

Differentiation is usually carried out in three main ways, **Symbolically**, 
**Numerically** and **Automatically**. We will now briefly overview each one of 
them.

## Symbolic differentiation

Symbolic differentiation involves the manipulation of mathematical expressions
to obtain exact dots. It is typically the only method taught in a calculus
course. While it leads to precise results, it can yield really long mathematical expressions, making it impractical for complex nested functions that can appear 
in different fields of science. Now consider the function

\begin{align}
f(x) = x^{4} + 3 x^{2} + 2 x
\end{align}

Symbolic differentiation is easy and quickly leads to 

\begin{align}
\frac{df(x)}{dx} =f'(x) =  4 x^{3} + 6 x + 2
\end{align}

Simple, but typically functions will be complex nested functions. Let us consider 
a manageable example that illustrates the point. Consider


\begin{align}
g(x) = x e^{-x^{2}}
\end{align}

Another simple function with a simple dot

\begin{align}
\frac{dg(x)}{dx} = g'(x)  = e^{-x^{2}} -  2 x^{2} e^{-x^{2}}
\end{align}

Using the chain rule

\begin{align}
\frac{df(g(x))}{dx} = f'(g(x)) g'(x)   
\end{align}

Then 

\begin{align}
\frac{df(g(x))}{dx} &= \left( 4 (x e^{-x^{2}})^{3} + 6 (x e^{-x^{2}}) + 2 \right) \left( e^{-x^{2}} -  2 x^{2} e^{-x^{2}} \right) \\
&= \left( 4 x^{3} e^{-x^{6}} + 6 x e^{-x^{2}} + 2 \right) e^{-x^{2}} \left( 1 -  2 x^{2}  \right) 
\end{align}

We can quickly appreciate how the expression becomes longer and longer as we compose functions, while it is not a problem in this case you can imagine how it would look as we compose more complex functions

## Numeric differentiation

Numeric differentiation tries to approximate the dot of a function, using 
the fact that the definition of a dot is 

\begin{align}
\frac{df(x)}{dx} = \lim_{h\to 0} \frac{f(x+h)-f(x)}{h} \approx \frac{f(x+h)-f(x)}{h}
\end{align}

with really small $h$, though it sounds simple, this naive recipe often fails due 
to truncation and round-off errors. Typically more stable formulas are used, but how to 
to this is beyond the scope of this article (see this other article  for a small intro and references).

Unfortunately, the computational cost scales poorly with the number of variables
https://www.youtube.com/watch?v=wG_nF1awSSY&t=305s
https://arxiv.org/abs/1502.05767
## Automatic differentiation


Automatic Differentiation (usually shortened to Autodiff) is a different 
alternative that computes exact dots efficiently (up to machine precision)
by applying the chain rule to the composite functions until only elementary
functions and operations remain. This simple idea resambles a lot what we just 
did to obtain analytical dots, the difference is that this time we will 
make the computer apply the chain rule in a specific way/

#### Autodiff Modes: Forward and Backward differentiation

Autodiff can be done in two ways, namely forward and reverse modes, each with 
their own advantages and disadvantages.

##### Forward Mode:

Forward mode (also known as left to right) computes directional dots 
alongside the function evaluation. It's particularly efficient for functions 
with few inputs and many outputs.

For a function $y=f(x)$ where both $x$ and $y$ are real forward mode computes 
the Jacobian-vector product on the side 

\begin{align}
 \dot{y} = J \dot{x}
\end{align}

where $\dot{x}$ is called the seed vector. 


In [ ]:
class autodiff:
    def __init__(self, value, dot=0):
        self.value = value
        self.dot= dot # works as dot{x}
    def __add__(self, other):
        try:
            res=autodiff(self.value +other.value)
            res.dot= self.dot +other.dot
            return res
        except:
            if other is None:
                other=0
            other=autodiff(other)
            res=autodiff(self.value +other.value)
            res.dot= self.dot +other.dot
            return res
    def __mul__(self, other):
        other=autodiff(other)
        res=autodiff(self.value*other.value)
        res.dot=self.value * other.dot + self.dot * other.value
        return res
    def __rmul__(self,other):
        other=autodiff(other)
        res=autodiff(self.value*other.value)
        res.dot=self.value * other.dot + self.dot * other.value
        return res
    def __pow__(self, n):
        res=autodiff(self.value ** n)
        res.dot= n * (self.value ** (n-1)) * self.dot
        return res
        
def f(x):
    return x**4 + 3*x**2 + 2*x

x = autodiff(2.0, 1.0)  # x = 2, dx/dx = 1 (~> dot{x})
result = f(x)

print(f"f(2) = {result.value}, f'(2) = {result.dot}")
# Output: f(2) = 42.0, f'(2) = 58.0


f(2) = 32.0, f'(2) = 46.0


https://kenndanielso.github.io/mlrefined/blog_posts/3_Automatic_differentiation/3_4_AD_forward_mode.html
Make a better implementation with more primitives. Understand the Jacobian bit

##### Backwards differentiation:

This is the main method used on most applications due to its popularity in machine learning
this popularity is due to the fact that there, they deal with functions with millions of inputs and few outputs, where this way of computing it excels. It computes derivatives by going backwards in the **computation graph** starting from the output and applying the chain rule until it has covered all of the graph.

In [65]:
class Node:
    def __init__(self, value):
        self.value = value
        self.grad = 0
        self._backward = lambda: None # this is defined as the forward mode is done based on the computation graph. 
        self._prev = set()

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.value + other.value)
        out._prev = {self, other}
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.value * other.value)
        out._prev = {self, other}
        def _backward():
            self.grad += other.value * out.grad
            other.grad += self.value * out.grad
        out._backward = _backward
        return out
    def __rmul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        out = Node(self.value * other.value)
        out._prev = {self, other}
        def _backward():
            self.grad += other.value * out.grad
            other.grad += self.value * out.grad
        out._backward = _backward
        return out
    def __pow__(self, n):
        out = Node(self.value ** n)
        out._prev = {self}

        def _backward():
            self.grad += n * (self.value ** (n-1)) * out.grad 

        out._backward = _backward
        return out

def backward(node):
    topo = []
    visited = set()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)

    build_topo(node)
    node.grad = 1
    for node in reversed(topo):
        node._backward()

# Example usage
x = Node(2.0)

y = x**4 + 3*x**2 + 2*x

backward(y)

print(f"f(2) = {y.value}, f'(2) = {x.grad}")
# Output: f(2) = 42.0, f'(2) = 58.0

f(2) = 32.0, f'(2) = 46.0


Generalize as well, do these in fortran and C as well